# IS 4487 Assignment 11: Predicting Airbnb Prices with Regression

In this assignment, you will:
- Load the Airbnb dataset you cleaned and transformed in Assignment 7
- Build a linear regression model to predict listing price
- Interpret which features most affect price
- Try to improve your model using only the most impactful predictors
- Practice explaining your findings to a business audience like a host, pricing strategist, or city partner

## Why This Matters

Pricing is one of the most important levers for hosts and Airbnb’s business teams. Understanding what drives price — and being able to predict it accurately — helps improve search results, revenue management, and guest satisfaction.

This assignment gives you hands-on practice turning a cleaned dataset into a predictive model. You’ll focus not just on code, but on what the results mean and how you’d communicate them to stakeholders.

<a href="https://colab.research.google.com/github/Stan-Pugsley/is_4487_base/blob/main/Assignments/assignment_11_regression.ipynb" target="_parent">
  <img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/>
</a>



## Original Source: Dataset Description

The dataset you'll be using is a **detailed Airbnb listing file**, available from [Inside Airbnb](https://insideairbnb.com/get-the-data/).

Each row represents one property listing. The columns include:

- **Host attributes** (e.g., host ID, host name, host response time)
- **Listing details** (e.g., price, room type, minimum nights, availability)
- **Location data** (e.g., neighborhood, latitude/longitude)
- **Property characteristics** (e.g., number of bedrooms, amenities, accommodates)
- **Calendar/booking variables** (e.g., last review date, number of reviews)

The schema is consistent across cities, so you can expect similar columns regardless of the location you choose.

In [1]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.linear_model import LinearRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error, r2_score


## 1. Load Your Transformed Airbnb Dataset

**Business framing:**  
Before building any models, we must start with clean, prepared data. In Assignment 7, you exported a cleaned version of your Airbnb dataset. You’ll now import that file for analysis.

### Do the following:
- Import your CSV file called `cleaned_airbnb_data_7.csv`.   (Note: If you had significant errors with assignment 7, you can use the file named "airbnb_listings.csv" in the DataSets folder on GitHub as a backup starting point.)
- Use `pandas` to load and preview the dataset


In [3]:
import pandas as pd

# Load the cleaned airbnb dataset
file_path = '/content/cleaned_airbnb_data_7.csv'
df = pd.read_csv(file_path)

# Preview the dataset
print(f"Dataset Shape: {df.shape}")
display(df.head())

Dataset Shape: (6734, 101)


,id,listing_url,scrape_id,last_scraped,source,name,description,picture_url,host_id,host_url,...,price_per_night,is_long_term_stay,room_type_Entire home/apt,room_type_Hotel room,room_type_Private room,room_type_Shared room,room_type_Entire home/apt.1,room_type_Hotel room.1,room_type_Private room.1,room_type_Shared room.1
0,103760,https://www.airbnb.com/rooms/103760,20260627225958,2026-06-28,city scrape,Central Centre 5 min walk to/from Central MTR,"Located right in the heart of Central, this 2 ...",https://a0.muscache.com/pictures/815221/056993...,304876,https://www.airbnb.com/users/show/304876,...,65.612222,0,True,False,False,False,True,False,False,False
1,163664,https://www.airbnb.com/rooms/163664,20260627225958,2026-06-28,city scrape,Soho off Hollywood Road P4,"Fun Fact: Possession St, is where the British ...",https://a0.muscache.com/pictures/15184030/8d06...,304876,https://www.airbnb.com/users/show/304876,...,41.102500,0,True,False,False,False,True,False,False,False
2,248140,https://www.airbnb.com/rooms/248140,20260627225958,2026-07-03,previous scrape,Bright Studio - Soho - Central HK,Our bright studio apartment is perfect for vis...,https://a0.muscache.com/pictures/4e5463bc-38be...,1300549,https://www.airbnb.com/users/show/1300549,...,NaN,0,True,False,False,False,True,False,False,False
3,263081,https://www.airbnb.com/rooms/263081,20260627225958,2026-06-29,city scrape,"3 睡房, 2500 平方呎 有工人 Family Friendly! 在中環半山 有車","Mid Level on the top of central, Next to the m...",https://a0.muscache.com/pictures/7348025/73278...,1370155,https://www.airbnb.com/users/show/1370155,...,2491.060000,0,False,False,True,False,False,False,True,False
4,306515,https://www.airbnb.com/rooms/306515,20260627225958,2026-06-28,city scrape,"HongKong,Central Bright Double Room",NaN,https://a0.muscache.com/pictures/3335548/00a8d...,1576511,https://www.airbnb.com/users/show/1576511,...,35.929000,0,True,False,False,False,True,False,False,False


### Reflection:
1. What business information does the dataset include?
2. How many rows and columns are present?

### ✍️ Your Response: 🔧
1. The business information that this dataset includes are lisitng charactersitics and the host profiles. It also tells us about pricing and booking statuses.

2. There are 6,734 rows and 101 columns.

## 2. Drop Columns Not Useful for Modeling

**Business framing:**  
Some columns — like post IDs or text — may not help us predict price and could add noise or bias.

### Do the following:
- Drop columns like `post_id`, `title`, `descr`, `details`, and `address` if they’re still in your dataset

In [4]:
# Drop columns that are text, URLs, or unique IDs which are not helpful for regression
columns_to_drop = [
    'id', 'listing_url', 'scrape_id', 'last_scraped', 'source', 'name',
    'description', 'picture_url', 'host_id', 'host_url', 'host_name',
    'host_since', 'host_location', 'host_about', 'host_thumbnail_url',
    'host_picture_url', 'host_neighbourhood', 'neighborhood_overview',
    'neighbourhood', 'neighbourhood_cleansed', 'neighbourhood_group_cleansed',
    'calendar_last_scraped', 'first_review', 'last_review'
]

# Filter columns_to_drop to only those that actually exist in the dataframe
existing_cols_to_drop = [col for col in columns_to_drop if col in df.columns]

df_cleaned = df.drop(columns=existing_cols_to_drop)

print(f"Original shape: {df.shape}")
print(f"New shape after dropping columns: {df_cleaned.shape}")
display(df_cleaned.head())

Original shape: (6734, 101)
New shape after dropping columns: (6734, 78)


,host_profile_id,host_profile_url,hosts_time_as_user_years,hosts_time_as_user_months,hosts_time_as_host_years,hosts_time_as_host_months,host_response_time,host_response_rate,host_acceptance_rate,host_is_superhost,...,price_per_night,is_long_term_stay,room_type_Entire home/apt,room_type_Hotel room,room_type_Private room,room_type_Shared room,room_type_Entire home/apt.1,room_type_Hotel room.1,room_type_Private room.1,room_type_Shared room.1
0,1462515188084590070,https://www.airbnb.com/users/profile/146251518...,15,6,15,5,NaN,NaN,NaN,f,...,65.612222,0,True,False,False,False,True,False,False,False
1,1462515188084590070,https://www.airbnb.com/users/profile/146251518...,15,6,15,5,NaN,NaN,NaN,f,...,41.102500,0,True,False,False,False,True,False,False,False
2,1462542170823986906,https://www.airbnb.com/users/profile/146254217...,14,8,14,8,NaN,NaN,NaN,f,...,NaN,0,True,False,False,False,True,False,False,False
3,1462543964897136608,https://www.airbnb.com/users/profile/146254396...,14,7,13,11,NaN,NaN,NaN,f,...,2491.060000,0,False,False,True,False,False,False,True,False
4,1462562133322138543,https://www.airbnb.com/users/profile/146256213...,14,5,14,5,NaN,NaN,NaN,f,...,35.929000,0,True,False,False,False,True,False,False,False


### Reflection:
1. What columns did you drop, and why?
2. What risks might occur if you included them in your model?

### ✍️ Your Response: 🔧
1. The columns I decided to drop are unique identifier columns like id and host id, as well as urls. I also dropped textual columns like name and description. I dropped these columns because they are not the standard numeric metrics (regarding the unique identifiers) as well as those textual columns are not simple categories.

2. If I were to include those columns in my model it may cause over fitting or also it can lead to incorrect matematical calculations because some numbers don't have a meaningful value if you go higher or lower, as well as text variables can't be used in a regression algorithm without more complicated help.

## 3. Explore Relationships Between Numeric Features

**Business framing:**  
Understanding how features relate to each other — and to the target — helps guide feature selection and modeling.

### Do the following:
- Generate a correlation matrix
- Identify which variables are strongly related to `price`

In [5]:
# Drop non-numeric identifier or URL columns that are still in df_cleaned to calculate correlation
numeric_df = df_cleaned.select_dtypes(include=['number'])

# Identify the exact price column name
price_col = 'price_per_night' if 'price_per_night' in numeric_df.columns else 'price'

# Generate correlation matrix
correlation_matrix = numeric_df.corr()

# Get correlation of all variables with price
price_correlation = correlation_matrix[price_col].sort_values(ascending=False)

print("Top Positive Correlations with Price:")
print(price_correlation.head(10))

print("\nTop Negative Correlations with Price:")
print(price_correlation.tail(10))

Top Positive Correlations with Price:
price_per_night                1.000000
price                          0.962139
price_quote_price_per_night    0.962139
log_price                      0.314010
price_quote_total_price        0.238156
accommodates                   0.181362
bedrooms                       0.092365
bathrooms                      0.083740
beds                           0.079176
estimated_revenue_l365d        0.070621
Name: price_per_night, dtype: float64

Top Negative Correlations with Price:
minimum_nights_avg_ntm                         -0.096996
calculated_host_listings_count_private_rooms   -0.104155
host_listings_count                            -0.105505
calculated_host_listings_count                 -0.112375
host_response_time                                   NaN
host_response_rate                                   NaN
host_acceptance_rate                                 NaN
host_total_listings_count                            NaN
host_verifications           

### Reflection:
1. Which variables had the strongest positive or negative correlation with price?
2. Which variables might be useful predictors?

### ✍️ Your Response: 🔧
1. The variables that have the strongest positive correlations are accomodates, bedrooms and bathrooms. The ones that have the strongest correlations with price are calculate_host_listings_count, host_listing count, and calculated_host_listings_count_private_rooms.

2. The variables that would be useful predictors are those with teh strongest positive correaltion, so accommodates, bedrooms, and bathrooms.

## 4. Define Features and Target Variable

**Business framing:**  
To build a regression model, you need to define what you’re predicting (target) and what you’re using to make that prediction (features).

### Do the following:
- Set `price` as your target variable
- Remove `price` from your predictors

In [6]:
# Define target column
target_col = 'price_per_night'

# Define columns to exclude from features (actual targets and price-leaking columns)
price_related_cols = [
    'price_per_night', 'price', 'price_quote_price_per_night',
    'price_quote_total_price', 'log_price', 'host_profile_id', 'host_profile_url'
]

# Select predictor features (only existing numeric columns that aren't price-related)
features_cols = [col for col in numeric_df.columns if col not in price_related_cols]

# Handle any missing values in the features to prevent modeling errors
X = numeric_df[features_cols].fillna(0)
y = numeric_df[target_col].fillna(0)

print(f"Features (X) shape: {X.shape}")
print(f"Target (y) shape: {y.shape}")
print("\nFeatures being used:")
print(list(X.columns))# Define target column
target_col = 'price_per_night'

# Define columns to exclude from features (actual targets and price-leaking columns)
price_related_cols = [
    'price_per_night', 'price', 'price_quote_price_per_night',
    'price_quote_total_price', 'log_price', 'host_profile_id', 'host_profile_url'
]

# Select predictor features (only existing numeric columns that aren't price-related)
features_cols = [col for col in numeric_df.columns if col not in price_related_cols]

# Handle any missing values in the features to prevent modeling errors
X = numeric_df[features_cols].fillna(0)
y = numeric_df[target_col].fillna(0)

print(f"Features (X) shape: {X.shape}")
print(f"Target (y) shape: {y.shape}")
print("\nFeatures being used:")
print(list(X.columns))# Define target column
target_col = 'price_per_night'

# Define columns to exclude from features (actual targets and price-leaking columns)
price_related_cols = [
    'price_per_night', 'price', 'price_quote_price_per_night',
    'price_quote_total_price', 'log_price', 'host_profile_id', 'host_profile_url'
]

# Select predictor features (only existing numeric columns that aren't price-related)
features_cols = [col for col in numeric_df.columns if col not in price_related_cols]

# Handle any missing values in the features to prevent modeling errors
X = numeric_df[features_cols].fillna(0)
y = numeric_df[target_col].fillna(0)

print(f"Features (X) shape: {X.shape}")
print(f"Target (y) shape: {y.shape}")
print("\nFeatures being used:")
print(list(X.columns))# Define target column
target_col = 'price_per_night'

# Define columns to exclude from features (actual targets and price-leaking columns)
price_related_cols = [
    'price_per_night', 'price', 'price_quote_price_per_night',
    'price_quote_total_price', 'log_price', 'host_profile_id', 'host_profile_url'
]

# Select predictor features (only existing numeric columns that aren't price-related)
features_cols = [col for col in numeric_df.columns if col not in price_related_cols]

# Handle any missing values in the features to prevent modeling errors
X = numeric_df[features_cols].fillna(0)
y = numeric_df[target_col].fillna(0)

print(f"Features (X) shape: {X.shape}")
print(f"Target (y) shape: {y.shape}")
print("\nFeatures being used:")
print(list(X.columns))# Define target column
target_col = 'price_per_night'

# Define columns to exclude from features (actual targets and price-leaking columns)
price_related_cols = [
    'price_per_night', 'price', 'price_quote_price_per_night',
    'price_quote_total_price', 'log_price', 'host_profile_id', 'host_profile_url'
]

# Select predictor features (only existing numeric columns that aren't price-related)
features_cols = [col for col in numeric_df.columns if col not in price_related_cols]

# Handle any missing values in the features to prevent modeling errors
X = numeric_df[features_cols].fillna(0)
y = numeric_df[target_col].fillna(0)

print(f"Features (X) shape: {X.shape}")
print(f"Target (y) shape: {y.shape}")
print("\nFeatures being used:")
print(list(X.columns))# Define target column
target_col = 'price_per_night'

# Define columns to exclude from features (actual targets and price-leaking columns)
price_related_cols = [
    'price_per_night', 'price', 'price_quote_price_per_night',
    'price_quote_total_price', 'log_price', 'host_profile_id', 'host_profile_url'
]

# Select predictor features (only existing numeric columns that aren't price-related)
features_cols = [col for col in numeric_df.columns if col not in price_related_cols]

# Handle any missing values in the features to prevent modeling errors
X = numeric_df[features_cols].fillna(0)
y = numeric_df[target_col].fillna(0)

print(f"Features (X) shape: {X.shape}")
print(f"Target (y) shape: {y.shape}")
print("\nFeatures being used:")
print(list(X.columns))

Features (X) shape: (6734, 51)
Target (y) shape: (6734,)

Features being used:
['hosts_time_as_user_years', 'hosts_time_as_user_months', 'hosts_time_as_host_years', 'hosts_time_as_host_months', 'host_response_time', 'host_response_rate', 'host_acceptance_rate', 'host_listings_count', 'host_total_listings_count', 'host_verifications', 'latitude', 'longitude', 'accommodates', 'bathrooms', 'bedrooms', 'beds', 'minimum_nights', 'maximum_nights', 'minimum_minimum_nights', 'maximum_minimum_nights', 'minimum_maximum_nights', 'maximum_maximum_nights', 'minimum_nights_avg_ntm', 'maximum_nights_avg_ntm', 'availability_30', 'availability_60', 'availability_90', 'availability_365', 'number_of_reviews', 'number_of_reviews_ltm', 'number_of_reviews_l30d', 'availability_eoy', 'number_of_reviews_ly', 'estimated_occupancy_l365d', 'estimated_revenue_l365d', 'review_scores_rating', 'review_scores_accuracy', 'review_scores_cleanliness', 'review_scores_checkin', 'review_scores_communication', 'review_scores


### Reflection:
1. What features are you using?
2. Why is this a regression problem and not a classification problem?

### ✍️ Your Response: 🔧
1. We are using numeric characteristics of the listing like accommodates, bedrooms, bathrooms and beds. We are excluding and direct price-leakage variables.

2. This is a regression problem because we are predicting a continuous, numerical value, which is the price per night of an Airbnb listing. If it were a classification problem, we would be trying to classify it into discrete, categorical buckets.


## 5. Split Data into Training and Testing Sets

### Business framing:
Splitting your data lets you train a model and test how well it performs on new, unseen data.

### Do the following:
- Use `train_test_split()` to split into 80% training, 20% testing



In [7]:
# Split into 80% training and 20% testing sets
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Print the shapes of the resulting splits
print(f"X_train shape: {X_train.shape}")
print(f"y_train shape: {y_train.shape}")
print(f"X_test shape: {X_test.shape}")
print(f"y_test shape: {y_test.shape}")

X_train shape: (5387, 51)
y_train shape: (5387,)
X_test shape: (1347, 51)
y_test shape: (1347,)


## 6. Fit a Linear Regression Model

### Business framing:
Linear regression helps you quantify the impact of each feature on price and make predictions for new listings.

### Do the following:
- Fit a linear regression model to your training data
- Use it to predict prices for the test set



In [8]:
# Initialize the Linear Regression model
model = LinearRegression()

# Fit the model to the training data
model.fit(X_train, y_train)

# Predict prices for the test set
y_pred = model.predict(X_test)

print("Linear Regression model has been successfully trained, and predictions have been generated!")

Linear Regression model has been successfully trained, and predictions have been generated!


## 7. Evaluate Model Performance

### Business framing:  
A good model should make accurate predictions. We’ll use Mean Squared Error (MSE) and R² to evaluate how close our predictions were to the actual prices.

### Do the following:
- Print MSE and R² score for your model

In [10]:
# Calculate Mean Squared Error (MSE) and R² Score
mse = mean_squared_error(y_test, y_pred)
r2 = r2_score(y_test, y_pred)

print(f"Mean Squared Error (MSE): {mse:.4f}")
print(f"R² Score: {r2:.4f}")

Mean Squared Error (MSE): 3369897.2584
R² Score: 0.1131


### Reflection:
1. What is your R² score? How well does your model explain price variation?
2. Is your MSE large or small? What could you do to improve it?

### ✍️ Your Response: 🔧
1. The R2 score is 0.1131. That means our model only explains about 11.31% of the variation in Airbnb prices per night. This is a very weak relationship and not super dependable.

2. The Mean Squared Error  is 3369897.26. This is pretty big. To improve this we can log-transform the target price variable so we can deal with the skewness and outliers.

## 8. Interpret Model Coefficients

### Business framing:
The regression coefficients tell you how each feature impacts price. This can help Airbnb guide hosts and partners.

### Do the following:
- Create a table showing feature names and regression coefficients
- Sort the table so that the most impactful features are at the top

In [11]:
# Create a DataFrame for feature names and coefficients
coefficients_df = pd.DataFrame({
    'Feature': X.columns,
    'Coefficient': model.coef_,
    'Absolute_Coefficient': [abs(c) for c in model.coef_]
})

# Sort the table so the most impactful features are at the top
coefficients_df = coefficients_df.sort_values(by='Absolute_Coefficient', ascending=False).reset_index(drop=True)

# Display the top 20 coefficients
display(coefficients_df.head(20))

,Feature,Coefficient,Absolute_Coefficient
0,latitude,1327.006339,1327.006339
1,is_long_term_stay,289.687252,289.687252
2,bathrooms,265.724583,265.724583
3,accommodates,160.991702,160.991702
4,longitude,-158.848800,158.848800
5,review_scores_value,-142.676636,142.676636
6,review_scores_cleanliness,138.473298,138.473298
7,review_scores_rating,128.749096,128.749096
8,reviews_per_month,-122.344094,122.344094
9,beds,-72.283634,72.283634


### Reflection:
1. Which features increased price the most?
2. Were any surprisingly negative?
3. What business insight could you draw from this?

### ✍️ Your Response: 🔧
1. The features that increased price the most were latitute, is_long_term_stay, bathrooms, and accommodates.

2. The reviews_per_month were negative and that surprised me.

3. A business insight that I can draw from this is that airbnbs should focus on what they control like adding more accommodations because that is highly correlated with price.


## 9. Try to Improve the Linear Regression Model

### Business framing:
The first version of your model included all available features — but not all features are equally useful. Removing weak or noisy predictors can often improve performance and interpretation.

### Do the following:
1. Choose your top 3–5 features with the strongest absolute coefficients
2. Rebuild the regression model using just those features
3. Compare MSE and R² between the baseline and refined model

In [12]:
# 1. Choose top features with the strongest absolute coefficients
top_features = ['latitude', 'is_long_term_stay', 'bathrooms', 'accommodates', 'longitude']

# 2. Subset the train and test data to only include these features
X_train_refined = X_train[top_features]
X_test_refined = X_test[top_features]

# Initialize and fit the refined model
refined_model = LinearRegression()
refined_model.fit(X_train_refined, y_train)

# Predict prices using the refined model
y_pred_refined = refined_model.predict(X_test_refined)

# 3. Calculate metrics for the refined model
mse_refined = mean_squared_error(y_test, y_pred_refined)
r2_refined = r2_score(y_test, y_pred_refined)

# Compare with baseline results
print("=== Model Comparison ===")
print(f"Baseline Model (All Features)  |  MSE: {mse:.4f}  |  R²: {r2:.4f}")
print(f"Refined Model (Top 5 Features) |  MSE: {mse_refined:.4f}  |  R²: {r2_refined:.4f}")

=== Model Comparison ===
Baseline Model (All Features)  |  MSE: 3369897.2584  |  R²: 0.1131
Refined Model (Top 5 Features) |  MSE: 3481054.4896  |  R²: 0.0838


### Reflection:
1. What features did you keep in the refined model, and why?
2. Did model performance improve? Why or why not?
3. Which model would you recommend to stakeholders?
4. How does this relate to your customized learning outcome you created in canvas?

### ✍️ Your Response: 🔧
1. The features I decided to keep in the refined model were latitutde, is_long_term_stay, bathrooms, accommodates, and longitude. These were chosen because they had the highest absolute coefficients in the baseline regression, so that means they have the strongest linear relationship with pricing.

2. Sadly, the model performance did not improves. The R2 score dropped from 0.1131 to 0.0838. The reason why this happened is probably because removing some features probably reduced the variance the linear model can explain.

3. The model I would recommend to stake holders is the baseline model because it has a little bit better predictive power. But honestly I wouldn't recommend either because we still have to handle outliers and the performance for prediciting is not high.

4. The way this relates to my customized learning outcome I created in canvas is because I want to help companies grow so being able to understand metrics and make predictions from the real world is important.

## 10. Reflect and Recommend

### Business framing:  
Ultimately, the value of your model comes from how well it can guide business decisions. Use your results to make real-world recommendations.

### Reflection:
1. What business question did your model help answer?
2. What would you recommend to Airbnb or its hosts?
3. What could you do next to improve this model or make it more useful?
4. How does this relate to your customized learning outcome you created in canvas?


### ✍️ Your Response: 🔧
1. The business question that my model helped answer is this: "What are factors that are the primary drivers of Airbnb listing prices, and how accurately can we predict prices using those basic listing characteristics?

2. I would recommend that hosts focus on metrics they can control and have high positive impacts so maximize the number of guest they can comfortably accomodate and adding or renovating bathrooms.

3. To improve the model I would handle the outliers and then do a log transformation.

4. This all relates to my customized learning outcome because I want to apply data-driven analytics to solve real-world problems.

## Submission Instructions
✅ Checklist:
- All code cells run without error
- All markdown responses are complete
- Submit on Canvas as instructed

In [13]:
!jupyter nbconvert --to html "assignment_11_regression.ipynb"

[NbConvertApp] Converting notebook assignment_11_regression.ipynb to html
[NbConvertApp] Writing 366284 bytes to assignment_11_regression.html
